# Phase 3 — Temporal Split

## Purpose
This notebook implements the leakage-safe temporal split for the Twitch re-engagement prediction task.

## Main task
Predict whether a user will re-engage with the same streamer within the next 3 days.

## Design decisions
- The dataset contains 6,148 timesteps at 10-minute resolution.
- The prediction horizon is 3 days = 432 timesteps.
- Temporal splitting must be chronological.
- Safety gaps are used to prevent label leakage across partitions.

## Planned outputs
- A split specification at timestep level
- A reusable temporal split table
- Sanity checks for leakage-safe partitioning

In [34]:
import pandas as pd

In [35]:
# Time structure constants for the Twitch dataset

TOTAL_TIMESTEPS = 6148
TIMESTEP_MINUTES = 10

TIMESTEPS_PER_HOUR = 60 // TIMESTEP_MINUTES
TIMESTEPS_PER_DAY = 24 * TIMESTEPS_PER_HOUR

PREDICTION_HORIZON_DAYS = 3
PREDICTION_HORIZON_TIMESTEPS = PREDICTION_HORIZON_DAYS * TIMESTEPS_PER_DAY

print("TOTAL_TIMESTEPS:", TOTAL_TIMESTEPS)
print("TIMESTEP_MINUTES:", TIMESTEP_MINUTES)
print("TIMESTEPS_PER_DAY:", TIMESTEPS_PER_DAY)
print("PREDICTION_HORIZON_TIMESTEPS:", PREDICTION_HORIZON_TIMESTEPS)

TOTAL_TIMESTEPS: 6148
TIMESTEP_MINUTES: 10
TIMESTEPS_PER_DAY: 144
PREDICTION_HORIZON_TIMESTEPS: 432


In [37]:
# Primary temporal split lengths

TRAIN_DAYS = 20
VALIDATION_DAYS = 7
GAP_DAYS = 3
FINAL_TAIL_DAYS = 3

TRAIN_TIMESTEPS = TRAIN_DAYS * TIMESTEPS_PER_DAY
VALIDATION_TIMESTEPS = VALIDATION_DAYS * TIMESTEPS_PER_DAY
GAP_TIMESTEPS = GAP_DAYS * TIMESTEPS_PER_DAY
FINAL_TAIL_TIMESTEPS = FINAL_TAIL_DAYS * TIMESTEPS_PER_DAY

print("TRAIN_TIMESTEPS:", TRAIN_TIMESTEPS)
print("VALIDATION_TIMESTEPS:", VALIDATION_TIMESTEPS)
print("GAP_TIMESTEPS:", GAP_TIMESTEPS)
print("FINAL_TAIL_TIMESTEPS:", FINAL_TAIL_TIMESTEPS)

TRAIN_TIMESTEPS: 2880
VALIDATION_TIMESTEPS: 1008
GAP_TIMESTEPS: 432
FINAL_TAIL_TIMESTEPS: 432


In [38]:
# Compute split boundaries

train_start = 0
train_end = TRAIN_TIMESTEPS

gap1_start = train_end
gap1_end = gap1_start + GAP_TIMESTEPS

validation_start = gap1_end
validation_end = validation_start + VALIDATION_TIMESTEPS

gap2_start = validation_end
gap2_end = gap2_start + GAP_TIMESTEPS

test_start = gap2_end
test_end = TOTAL_TIMESTEPS - FINAL_TAIL_TIMESTEPS

tail_start = test_end
tail_end = TOTAL_TIMESTEPS

print("Train:", train_start, train_end)
print("Gap 1:", gap1_start, gap1_end)
print("Validation:", validation_start, validation_end)
print("Gap 2:", gap2_start, gap2_end)
print("Test:", test_start, test_end)
print("Final tail:", tail_start, tail_end)

Train: 0 2880
Gap 1: 2880 3312
Validation: 3312 4320
Gap 2: 4320 4752
Test: 4752 5716
Final tail: 5716 6148


In [39]:
# Create a dataframe with one row per timestep
temporal_split = pd.DataFrame({
    "timestep_id": range(TOTAL_TIMESTEPS)
})

# Assign segment labels
temporal_split["segment"] = "unassigned"

temporal_split.loc[
    (temporal_split["timestep_id"] >= train_start) &
    (temporal_split["timestep_id"] < train_end),
    "segment"
] = "train"

temporal_split.loc[
    (temporal_split["timestep_id"] >= gap1_start) &
    (temporal_split["timestep_id"] < gap1_end),
    "segment"
] = "gap_1"

temporal_split.loc[
    (temporal_split["timestep_id"] >= validation_start) &
    (temporal_split["timestep_id"] < validation_end),
    "segment"
] = "validation"

temporal_split.loc[
    (temporal_split["timestep_id"] >= gap2_start) &
    (temporal_split["timestep_id"] < gap2_end),
    "segment"
] = "gap_2"

temporal_split.loc[
    (temporal_split["timestep_id"] >= test_start) &
    (temporal_split["timestep_id"] < test_end),
    "segment"
] = "test"

temporal_split.loc[
    (temporal_split["timestep_id"] >= tail_start) &
    (temporal_split["timestep_id"] < tail_end),
    "segment"
] = "final_tail"

temporal_split.head(10)

,timestep_id,segment
0,0,train
1,1,train
2,2,train
3,3,train
4,4,train
5,5,train
6,6,train
7,7,train
8,8,train
9,9,train


In [40]:
# Check segment counts
segment_counts = temporal_split["segment"].value_counts().sort_index()
print(segment_counts)

# Check total number of rows
print("\nTotal rows:", len(temporal_split))

# Check whether any timestep is still unassigned
print("Number of unassigned timesteps:", (temporal_split["segment"] == "unassigned").sum())

segment
final_tail     432
gap_1          432
gap_2          432
test           964
train         2880
validation    1008
Name: count, dtype: int64

Total rows: 6148
Number of unassigned timesteps: 0


In [41]:
# Mark whether a timestep can be used as a prediction anchor
anchor_segments = ["train", "validation", "test"]

temporal_split["is_anchor_candidate"] = temporal_split["segment"].isin(anchor_segments)

temporal_split.head(10)

print("Anchor candidate timesteps:", temporal_split["is_anchor_candidate"].sum())
print("Non-anchor timesteps:", (~temporal_split["is_anchor_candidate"]).sum())

Anchor candidate timesteps: 4852
Non-anchor timesteps: 1296


In [42]:
# Inspect rows around split boundaries

boundary_preview = pd.concat([
    temporal_split.iloc[2877:2883],   # train -> gap_1
    temporal_split.iloc[3309:3315],   # gap_1 -> validation
    temporal_split.iloc[4317:4323],   # validation -> gap_2
    temporal_split.iloc[4749:4755],   # gap_2 -> test
    temporal_split.iloc[5713:5719],   # test -> final_tail
])

boundary_preview

,timestep_id,segment,is_anchor_candidate
2877,2877,train,True
2878,2878,train,True
2879,2879,train,True
2880,2880,gap_1,False
2881,2881,gap_1,False
2882,2882,gap_1,False
3309,3309,gap_1,False
3310,3310,gap_1,False
3311,3311,gap_1,False
3312,3312,validation,True


In [43]:
# Save temporal split table
output_path = "../data_processed/temporal_split.csv"

temporal_split.to_csv(output_path, index=False)

print(f"Saved temporal split table to: {output_path}")

Saved temporal split table to: ../data_processed/temporal_split.csv


In [44]:
# Reload the saved split artifact
split_path = "../data_processed/temporal_split.csv"

temporal_split_loaded = pd.read_csv(split_path)

temporal_split_loaded.head()

,timestep_id,segment,is_anchor_candidate
0,0,train,True
1,1,train,True
2,2,train,True
3,3,train,True
4,4,train,True


In [45]:
print("Rows:", len(temporal_split_loaded))
print("Columns:", temporal_split_loaded.columns.tolist())

Rows: 6148
Columns: ['timestep_id', 'segment', 'is_anchor_candidate']


In [46]:
interactions_path = "../data_processed/gold_100k.csv"
interactions = pd.read_csv(interactions_path)

print("Rows in dataset:", len(interactions))
interactions.head()

Rows in dataset: 3051733


,user_id,stream_id,streamer_name,start_time,stop_time,duration_intervals,duration_minutes
0,1,33842865744,mithrain,154,156,2,20
1,1,33846768288,alptv,166,169,3,30
2,1,33886469056,mithrain,587,588,1,10
3,1,33887624992,wtcn,589,591,2,20
4,1,33890145056,jrokezftw,591,594,3,30


In [47]:
print("Min start_time:", interactions["start_time"].min())
print("Max start_time:", interactions["start_time"].max())

print("Min stop_time:", interactions["stop_time"].min())
print("Max stop_time:", interactions["stop_time"].max())

Min start_time: 0
Max start_time: 6147
Min stop_time: 1
Max stop_time: 6148


## Join viewing sessions with the temporal split table

The processed Twitch dataset is structured at the session level, where each row represents a viewing interval with a `start_time` and `stop_time`.

At this stage, the temporal split table is joined to the session data using `start_time` as the anchor timestamp. This reflects the methodological decision that each prediction instance is defined at the start of a viewing session.

As a result, each session receives:
- a temporal segment label (`train`, `gap_1`, `validation`, `gap_2`, `test`, or `final_tail`)
- an indicator showing whether its `start_time` is eligible to serve as a prediction anchor

In [48]:
# Join temporal split information to viewing sessions using start_time

anchors = interactions.merge(
    temporal_split,
    left_on="start_time",
    right_on="timestep_id",
    how="left"
)

anchors.head()

,user_id,stream_id,streamer_name,start_time,stop_time,duration_intervals,duration_minutes,timestep_id,segment,is_anchor_candidate
0,1,33842865744,mithrain,154,156,2,20,154,train,True
1,1,33846768288,alptv,166,169,3,30,166,train,True
2,1,33886469056,mithrain,587,588,1,10,587,train,True
3,1,33887624992,wtcn,589,591,2,20,589,train,True
4,1,33890145056,jrokezftw,591,594,3,30,591,train,True


In [55]:
# Validate the join with temporal_split

print("Rows in anchors:", len(anchors))
print("Missing timestep_id:", anchors["timestep_id"].isna().sum())
print("Missing segment:", anchors["segment"].isna().sum())
print("Missing is_anchor_candidate:", anchors["is_anchor_candidate"].isna().sum())

print("\nSegment distribution after join:")
print(anchors["segment"].value_counts())

Rows in anchors: 3051733
Missing timestep_id: 0
Missing segment: 0
Missing is_anchor_candidate: 0

Segment distribution after join:
segment
train         1373160
validation     520761
test           486650
final_tail     230991
gap_2          223085
gap_1          217086
Name: count, dtype: int64


## Label definition: 3-day re-engagement

The prediction target is defined as whether a user returns to the same streamer within a 3-day horizon after the start of a viewing session.

For an anchor session `(user_id, streamer_name, start_time)`, the label is defined as:

label = 1 if another session exists with the same user and streamer such that:

start_time < next_start_time ≤ start_time + 432

where 432 timesteps correspond to 3 days.

Otherwise, the label is set to 0.

This definition ensures that the anchor session itself is not counted as a return event and that only future interactions are considered.

In [61]:
# Sort anchors and identify the next session for each user-stream pair

anchors = anchors.sort_values(
    by=["user_id", "streamer_name", "start_time"]
).copy()

anchors["next_start_time"] = anchors.groupby(
    ["user_id", "streamer_name"]
)["start_time"].shift(-1)

anchors[["user_id", "streamer_name", "start_time", "next_start_time", "segment"]].head(10)

,user_id,streamer_name,start_time,next_start_time,segment
1,1,alptv,166,NaN,train
5,1,berkriptepe,734,NaN,train
13,1,elraenn,2600,4314.0,train
25,1,elraenn,4314,NaN,validation
26,1,eraymaskulen,4327,NaN,gap_2
38,1,esl_csgo,5208,NaN,test
39,1,grimnax,5322,5415.0,test
44,1,grimnax,5415,NaN,test
31,1,h3x_tv,4859,NaN,test
19,1,jahrein,3757,4865.0,validation


## Filter prediction anchor candidates

A session can serve as a prediction anchor only if its `start_time` falls within one of the following segments:

- train
- validation
- test

Sessions starting within the safety gaps or the final tail period are excluded because their label windows may overlap with future data or fall outside the observable time range.

The resulting dataset contains all valid prediction anchors for the supervised learning task.

In [62]:
# Keep only anchor-eligible sessions

anchor_dataset = anchors[anchors["is_anchor_candidate"]].copy()

print("Total prediction anchors:", len(anchor_dataset))

anchor_dataset.head()

Total prediction anchors: 2380571


,user_id,stream_id,streamer_name,start_time,stop_time,duration_intervals,duration_minutes,timestep_id,segment,is_anchor_candidate,next_start_time
1,1,33846768288,alptv,166,169,3,30,166,train,True,NaN
5,1,33903958784,berkriptepe,734,737,3,30,734,train,True,NaN
13,1,34079135968,elraenn,2600,2601,1,10,2600,train,True,4314.0
25,1,34236660832,elraenn,4314,4315,1,10,4314,validation,True,NaN
38,1,34328509984,esl_csgo,5208,5209,1,10,5208,test,True,NaN


In [63]:
# Compute the 3-day re-engagement label

anchor_dataset["label"] = (
    (anchor_dataset["next_start_time"] > anchor_dataset["start_time"]) &
    (anchor_dataset["next_start_time"] <= anchor_dataset["start_time"] + PREDICTION_HORIZON_TIMESTEPS)
).astype(int)

anchor_dataset[["user_id", "stream_id", "start_time", "next_start_time", "label"]].head(10)

,user_id,stream_id,start_time,next_start_time,label
1,1,33846768288,166,NaN,0
5,1,33903958784,734,NaN,0
13,1,34079135968,2600,4314.0,0
25,1,34236660832,4314,NaN,0
38,1,34328509984,5208,NaN,0
39,1,34334577616,5322,5415.0,1
44,1,34347669376,5415,NaN,0
31,1,34293733056,4859,NaN,0
19,1,34188931888,3757,4865.0,0
33,1,34294941584,4865,5310.0,0


In [64]:
anchor_dataset["label"].mean()

0.2279537136258486

In [65]:
anchor_dataset.groupby("segment")["label"].mean()

segment
test          0.239721
train         0.219275
validation    0.239843
Name: label, dtype: float64

In [66]:
# Define prediction timestamp (anchor time)

anchor_dataset["prediction_time"] = anchor_dataset["start_time"]

anchor_dataset.head()

,user_id,stream_id,streamer_name,start_time,stop_time,duration_intervals,duration_minutes,timestep_id,segment,is_anchor_candidate,next_start_time,label,prediction_time
1,1,33846768288,alptv,166,169,3,30,166,train,True,NaN,0,166
5,1,33903958784,berkriptepe,734,737,3,30,734,train,True,NaN,0,734
13,1,34079135968,elraenn,2600,2601,1,10,2600,train,True,4314.0,0,2600
25,1,34236660832,elraenn,4314,4315,1,10,4314,validation,True,NaN,0,4314
38,1,34328509984,esl_csgo,5208,5209,1,10,5208,test,True,NaN,0,5208


In [68]:
print(anchor_dataset.columns)

Index(['user_id', 'stream_id', 'streamer_name', 'start_time', 'stop_time',
       'duration_intervals', 'duration_minutes', 'timestep_id', 'segment',
       'is_anchor_candidate', 'next_start_time', 'label', 'prediction_time'],
      dtype='object')


In [71]:
# Select columns for the final modeling dataset

model_dataset = anchor_dataset[
    [
        "user_id",
        "streamer_name",
        "stream_id",
        "prediction_time",
        "duration_intervals",
        "duration_minutes",
        "segment",
        "label"
    ]
].copy()

print("Rows:", len(model_dataset))
print("Columns:", model_dataset.columns.tolist())

model_dataset.shape

Rows: 2380571
Columns: ['user_id', 'streamer_name', 'stream_id', 'prediction_time', 'duration_intervals', 'duration_minutes', 'segment', 'label']


(2380571, 8)

## Save the final modeling dataset

After defining the leakage-safe temporal split and constructing the 3-day re-engagement label, the final session-level modeling dataset is saved for downstream experiments.

This dataset contains:
- entity identifiers (`user_id`, `streamer_name`, `stream_id`)
- the prediction timestamp
- session duration information
- the temporal segment assignment
- the binary re-engagement label

This saved file will serve as the main input for feature engineering and model training in later phases.

In [72]:
# Save the final modeling dataset

model_output_path = "../data_processed/model_dataset_100k_3day.csv"

model_dataset.to_csv(model_output_path, index=False)

print(f"Saved modeling dataset to: {model_output_path}")

Saved modeling dataset to: ../data_processed/model_dataset_100k_3day.csv


## Final sanity checks for the modeling dataset

Before closing Phase 3, the final modeling dataset is checked at the segment level.

The goal is to confirm that:
- the number of rows per segment matches expectations
- the positive label rate remains reasonably stable across train, validation, and test

These checks help verify that the temporal split and label construction pipeline behaved as intended.

In [73]:
# Final sanity checks for the saved modeling dataset

print("Row counts by segment:")
print(model_dataset["segment"].value_counts())

print("\nPositive rate by segment:")
print(model_dataset.groupby("segment")["label"].mean())

print("\nOverall positive rate:")
print(model_dataset["label"].mean())

Row counts by segment:
segment
train         1373160
validation     520761
test           486650
Name: count, dtype: int64

Positive rate by segment:
segment
test          0.239721
train         0.219275
validation    0.239843
Name: label, dtype: float64

Overall positive rate:
0.2279537136258486


## Temporal split completion

This phase established the leakage-safe temporal evaluation framework for the Twitch re-engagement prediction task.

- A chronological temporal split was defined at the timestep level.
- Safety gaps were introduced between segments to prevent label leakage.
- Viewing sessions were joined with the temporal split table using the session start time.
- Prediction anchors were restricted to sessions occurring in the train, validation, and test segments.
- The target variable was defined as whether a user returns to the same streamer within a 3-day horizon.
- Labels were constructed using only future interactions relative to the prediction timestamp.

The resulting modeling dataset contains 2,380,571 prediction anchors and will serve as the main input for feature engineering and model training in subsequent phases.